# Week 7 Exercise: QLoRA Fine-tuned Model Evaluation

This notebook loads a pre-trained Llama-3.2-3B model with QLoRA adapters and evaluates it on the items_prompts test set. Designed for Google Colab.

**Run in Colab (GPU):** [Open in Google Colab](https://colab.research.google.com/drive/1peU3XpnHNWhoTKLeo_w7E2kcGExt1hhH?usp=sharing)

### Environment setup

Install dependencies and download `util.py` from the course repo. Uncomment the lines below when running in Colab.

In [ ]:
# !pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1
# !wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

### Imports

Libraries for model loading, quantization, PEFT, datasets, and the custom evaluation helper.

In [ ]:
# imports
import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
from peft import PeftModel
from util import evaluate

### Constants

Base model, project name, dataset choice (lite vs full), and QLoRA settings (4-bit/8-bit, bf16).

In [ ]:
# Constants

BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price"
HF_USER = "ed-donner"

LITE_MODE = True

DATA_USER = "ed-donner"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

if LITE_MODE:
  RUN_NAME = "2025-11-30_15.10.55-lite"
  REVISION = None
else:
  RUN_NAME = "2025-11-28_18.47.07"
  REVISION = "b19c8bfea3b6ff62237fbb0a8da9779fc12cefbd"

PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"


# Hyper-parameters - QLoRA

QUANT_4_BIT = True
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

### Hugging Face login

Log in using your Hugging Face token (e.g. from Colab secrets as `HF_TOKEN`).


In [ ]:
# Log in to HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

### Datasets

Load the items_prompts dataset from the Hub and use the test split for evaluation.


### Quantization config

Choose 4-bit (NF4) or 8-bit quantization for loading the base model; bf16 is used when the GPU supports it.

In [ ]:
dataset = load_dataset(DATASET_NAME)
test = dataset['test']

### Load tokenizer and model

Load the base model with quantization, then attach the PEFT adapter from the Hub. Prints memory footprint.

In [ ]:
# pick the right quantization

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )

In [ ]:
# Load the Tokenizer and the Model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# Load the fine-tuned model with PEFT
if REVISION:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=REVISION)
else:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)


print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

**Prediction function**

Defines `model_predict(item)` to encode the prompt, generate up to 8 new tokens, and return the decoded text.

### Evaluation

Run inference on the test set and compute metrics using the custom `evaluate` helper.

**Run evaluation**

Set seed for reproducibility and call `evaluate(model_predict, test)` to score the model on the test set.

In [ ]:
def model_predict(item):
    inputs = tokenizer(item["prompt"],return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

In [ ]:
set_seed(42)
evaluate(model_predict, test)